In [ ]:
import pandas as pd
import seaborn as sns

In [ ]:
data_path = "../data/Customer-Churn-Records.csv"

df = pd.read_csv(data_path)
df.drop(columns=["RowNumber", "CustomerId", "Surname"], inplace=True)
df.head()

In [ ]:
corr = df[["Complain", "Exited"]].corr()
sns.heatmap(corr, vmax=1, annot=True, square=True)

In [ ]:
corr

Given that the "Complain" feature is highly correlated to the target "Exited", it is best to remove it from the features as it might be a data leak

In [ ]:
df.drop(columns=["Complain"], inplace=True)

Based on the `initial_report.html`, the observations are that:
- "CreditScore" is bimodal distributed with lots of outliers at 850.
- "Age" is right-skewed, with mean of 38.9 and median of 37, indicating that the dataset concentrates more on early to middle adulthood.
- "Balance" is also bimodal distributed with more than 36% at 0.
- "NumOfProducts" is mostly 1 and 2 with very few at 3 or 4.
- "EstimatedSalary" is uniform distributed as "Point Earned".
- Target "Exited" is not balanced with 79.62% not exited, a baseline at 80% accuracy
- No feature has null value

# Feature importance

## Churn rate and risk ratio

In [ ]:
tot_churn = df.Exited.mean()

In [ ]:
from IPython.display import display

cat = [
    "Geography",
    "Gender",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "Satisfaction Score",
    "Card Type",
]

# Churn rate and risk ratio by each categorical feature
for c in cat:
    df_group = df.groupby(c).Exited.agg(["mean"])
    df_group["risk_ratio"] = df_group["mean"] / tot_churn
    display(df_group)

Based on the tables above, customers with more than 2 products are most certainly to churn.

## Mutual information

In [ ]:
from sklearn.metrics import mutual_info_score


def mi_churn_score(series):
    return mutual_info_score(series, df.Exited)


mi = df[cat].apply(mi_churn_score)
mi.sort_values(ascending=False)